In [5]:
from sklearn.model_selection import train_test_split
from pyoperon.sklearn import SymbolicRegressor  # pip install pyoperon
from sympy.printing.pycode import pycode
from sklearn.metrics import r2_score
import pandas as pd
import matplotlib as plt
import sympy as sp
import numpy as np
import textwrap

class SR:
    def __init__(self, x_values, y_values, x_true, y_true, optimizer_iterations=10, max_length=15, population_size=1000, generations=100, seed=42):
        self.X_train = x_values
        self.y_train = y_values
        self.X_test = x_true
        self.y_test = y_true
        self.optimizer_iterations = optimizer_iterations
        self.max_length = max_length
        self.population_size = population_size
        self.generations = generations
        self.seed = seed

        # Splitting data into train and test data
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(x_values, y_values, test_size=0.2, random_state=self.seed)

        # Create a SymbolicRegressor object and use its sklearn-like interface
        self.reg = SymbolicRegressor(
                allowed_symbols='add,sub,mul,div,pow,constant,variable',
                optimizer_iterations=self.optimizer_iterations,
                max_length=self.max_length,
                n_threads=8,
                objectives = ['r2'],
                population_size=self.population_size,
                generations=self.generations,
                random_state=self.seed
            )
        
    def train(self):
        self.reg.fit(self.X_train, self.y_train)
        best_model = self.reg.get_model_string(self.reg.model_, precision=5) 
        self.best_model_simple = sp.simplify(best_model) 

        print("R2 train:   ", self.reg.score(self.X_train,self.y_train))
        print("R2 test:    ", self.reg.score(self.X_test,self.y_test))
        # print("R2 train:   ", r2_score(self.y_train,self.reg.predict(self.X_train)))
        # print("R2 test:    ", r2_score(self.y_test,self.reg.predict(self.X_test)))
        print("Best model: ", best_model)
        print("Best model simple: ", self.best_model_simple)

        return self.best_model_simple
    
    def make_symbolic_function(self): 
        variables = ["X1", "X2", "X3"] 
        expr = self.best_model_simple
        expr = str(expr)
        expr = expr.replace("^", "**")
        expr = expr.replace("sqrt", "np.sqrt")
        expr = expr.replace("sin", "np.sin")
        expr = expr.replace("cos", "np.cos")
        expr = expr.replace("tan", "np.tan")
        expr = expr.replace("log", "np.log")
        expr = expr.replace("exp", "np.exp")

        # Safe eval with numpy
        def func(**kwargs):
            return eval(expr, {"__builtins__": {},"np": np}, kwargs)

        return func

    def predict(self, x):
        func = self.make_symbolic_function()
        return func(X1=x[0],X2=x[1],X3=x[2])

    def save_numpy_predict(self, path="predict_fn.py"):    
        expr = self.best_model_simple
        X1, X2, X3 = sp.symbols("X1 X2 X3")
        body = pycode(expr)
        body = (body.replace("sin", "np.sin")
                    .replace("cos", "np.cos")
                    .replace("tan", "np.tan")
                    .replace("log", "np.log")
                    .replace("exp", "np.exp")
                    .replace("sqrt", "np.sqrt"))

        source = textwrap.dedent(f"""\
            class model:
                def predict(x):
                    X1, X2, X3 = x[:,0], x[:,1], x[:,2]
                    return {body}
            """)
        with open(path, "w", encoding="utf-8") as f:
            f.write(source)

In [6]:
df = pd.read_csv("/home/simonmadsen/SummerSchool26/international-summer-school-robotics-TER-UR/case 2/data/test-4.csv")

target_current = df["target_current1"][:10000]
actual_current = df["target_current1"][:10000]

# Training
df = pd.read_csv("/home/simonmadsen/SummerSchool26/international-summer-school-robotics-TER-UR/case 2/data/test-4.csv")

cols_x = ["target_q0","target_q1","target_q2","target_q3","target_q4","target_q5","target_qd0","target_qd1","target_qd2","target_qd3","target_qd4","target_qd5","target_qdd0","target_qdd1","target_qdd2","target_qdd3","target_qdd4","target_qdd5","target_current0","target_current1","target_current2","target_current3","target_current4","target_current5","target_moment0","target_moment1","target_moment2","target_moment3","target_moment4","target_moment5","vel","acc"]
cols_x = ["target_current1","vel","acc"]
X = df[cols_x][:10000]

cols_y = ["actual_current1"]
y = df[cols_y][:10000]

sr = SR(X,y,None,None)
sr.train()

/home/simonmadsen/miniforge3/envs/datasim/lib/python3.12/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


R2 train:    0.9826153516769409
R2 test:     0.9830030798912048
Best model:  (0.00034 + (1.00000 * ((((-0.00010) * X3) + (1.08165 * X1)) - ((((35.80779 * X1) - (0.39924 ^ (0.43781 * X1))) * ((0.00996 * X1) + 0.31676)) * ((-0.00002) * X1)))))
Best model simple:  -2.0e-5*X1*(0.39924**(0.43781*X1) - 35.80779*X1)*(0.00996*X1 + 0.31676) + 1.08165*X1 - 0.0001*X3 + 0.00034


-2.0e-5*X1*(0.39924**(0.43781*X1) - 35.80779*X1)*(0.00996*X1 + 0.31676) + 1.08165*X1 - 0.0001*X3 + 0.00034

In [ ]:
X1 = sp.symbols('X1')
f = sp.lambdify(X1, sr.best_model_simple, 'numpy')
predicted_current = f(target_current)